In [ ]:
import numpy as np
from scipy.stats import norm
import math
import matplotlib.pyplot as plt
import pandas as pd

from scipy.optimize import brentq,minimize_scalar

class GaussianCopula:
    def __init__(self, M, R, rho, T, delta, n_herm):
        self.M = M
        self.R = R
        self.rho = rho
        self.T = T
        self.delta = delta
        self.n_herm = n_herm

        self.nodes, self.weights = np.polynomial.hermite.hermgauss(n_herm)

        self.N = int(self.T / self.delta)

        self.times = np.arange(0, self.N + 1) * self.delta

        #Load rates
        df = pd.read_excel(r"C:\Speciale\rates_20-06-2025.xlsx")
        rates = []
        for i in range(1,len(df.columns)):
            rates.append(df.iloc[0,i])

        #Load index spreads
        df = pd.read_excel(r"C:\Speciale\index_spread_20-06-2025.xlsx")
        index_spread = (df.iloc[0, 1:])/10000
        index_spread = index_spread.tolist()
        self.index_spread = index_spread

        #Calculate default probabilities
        rate_times = np.arange(0, len(rates)) * 0.125
        self.zcb = np.exp(-np.array(rates)/100 * rate_times)

        default_probs = np.zeros(self.N+1)

        default_probs = np.zeros(self.N+1)
        maturities = [0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5]
        hazard_rates = self.bootstrap_hazard_rates(index_spread,maturities)
        h1 = hazard_rates[0]
        h2 = hazard_rates[1]
        h3 = hazard_rates[2]
        h4 = hazard_rates[3]
        h5 = hazard_rates[4]
        h6 = hazard_rates[5]
        h7 = hazard_rates[6]
        h8 = hazard_rates[7]
        h9 = hazard_rates[8]
        h10 = hazard_rates[9]

        for i in range(3):
            default_probs[i] = 1-np.exp(-h1*i*self.delta)
        for i in range(3,5):
            default_probs[i] = 1-(1-default_probs[2])*np.exp(-h2*(i-2)*self.delta)
        for i in range(5,7):
            default_probs[i] = 1-(1-default_probs[4])*np.exp(-h3*(i-4)*self.delta)
        for i in range(7,9):
            default_probs[i] = 1-(1-default_probs[6])*np.exp(-h4*(i-6)*self.delta)
        for i in range(9,11):
            default_probs[i] = 1-(1-default_probs[8])*np.exp(-h5*(i-8)*self.delta)
        for i in range(11,13):
            default_probs[i] = 1-(1-default_probs[10])*np.exp(-h6*(i-10)*self.delta)
        for i in range(13,15):
            default_probs[i] = 1-(1-default_probs[12])*np.exp(-h7*(i-12)*self.delta)
        for i in range(15,17):
            default_probs[i] = 1-(1-default_probs[14])*np.exp(-h8*(i-14)*self.delta)
        for i in range(17,19):
            default_probs[i] = 1-(1-default_probs[16])*np.exp(-h9*(i-16)*self.delta)
        for i in range(19,21):
            default_probs[i] = 1-(1-default_probs[18])*np.exp(-h10*(i-18)*self.delta)

        self.default_probs = default_probs
        print(default_probs)
        
        self.default_barriers = norm.ppf(np.clip(default_probs, 1e-12,1 - 1e-12))

    def bootstrap_hazard_rates(self, spreads, maturities):

        hazard_rates = []

        def survival(t, hazard_rates, maturities):
            total = 0.0
            prev_T = 0.0

            for h, T in zip(hazard_rates, maturities):
                if t <= T:
                    total += h * (t - prev_T)
                    return np.exp(-total)
                else:
                    total += h * (T - prev_T)
                    prev_T = T

            return np.exp(-total)

        for k in range(len(spreads)):

            s_k = spreads[k]
            T_k = maturities[k]

            def cds_equation(hk):

                temp_hazards = hazard_rates + [hk]
                temp_maturities = maturities[:k+1]

                total = 0.0

                N = int(T_k / self.delta)

                for i in range(1, N + 1):
                    Ti = i * self.delta
                    Ti_m1 = (i - 1) * self.delta

                    # discount factor
                    P = self.zcb[2*i]

                    # survival probabilities
                    S_i = survival(Ti, temp_hazards, temp_maturities)
                    S_im1 = survival(Ti_m1, temp_hazards, temp_maturities)

                    # CDS equation 
                    total += P * (
                        (1 - self.R) * (S_im1 - S_i)
                        - s_k * self.delta * (S_im1 + S_i) / 2
                    )

                return total

            # solve for hk
            hk = brentq(cds_equation, 1e-6, 5.0)
            hazard_rates.append(hk)

        return hazard_rates        
    
    def calc_default_barrier(self,t):
        default_barrier = norm.ppf(self.calc_default_probs(t))
        return default_barrier
    
    def integrate_gauss_hermite(self, func):
        nodes = self.nodes
        weights = self.weights
        
        total = 0.0
        for i in range(self.n_herm):
            S = np.sqrt(2) * nodes[i]
            total += weights[i] * func(S)
        
        return total / np.sqrt(np.pi)
    
    def calc_expected_tranche_principal(self, A, B, S, k):
        S = np.asarray(S).item()
        S = float(S)
        # conditional default probability
        barrier = self.default_barriers[k]

        sqrt_rho = np.sqrt(self.rho)
        sqrt_1mr = np.sqrt(1-self.rho)

        p = norm.cdf((barrier - sqrt_rho*S) / sqrt_1mr)
        p = min(max(p, 1e-16), 1-1e-16)

        M = self.M
        R = self.R

        nl = A * M / (1 - R)
        nh = B * M / (1 - R)

        il = min(math.ceil(nl), M+1)
        ih = min(math.ceil(nh), M+1)

        # compute binomial probabilities recursively
        probs = np.zeros(M+1)
        probs[0] = (1 - p)**M

        ratio = p/(1-p)

        for n in range(M):
            probs[n+1] = probs[n] * (M-n)/(n+1) * ratio

        sum1 = probs[:il].sum()

        sum2 = 0.0
        for j in range(il, ih):
            loss_fraction = (B - j*(1-R)/M)/(B-A)
            sum2 += probs[j] * loss_fraction

        return sum1 + sum2

    def calc_PV_expected_regular_spread_payments(self,A,B):  #A in Hull
        delta = self.delta # assume equidistant payments
        def integrand(S): #conditional expected payments * density
            N = self.N
            sum = 0
            for k in range(1, N+1):
                sum += delta * self.calc_expected_tranche_principal(A,B,S,k) * self.zcb[2*k]
            return sum
        
        return self.integrate_gauss_hermite(integrand)
        
        
    def calc_accrual_payment(self,A,B): #B in Hull
        delta = self.delta # assume equidistant payments
        
        def integrand(S): #conditional accrual payments * density
            N = self.N
            sum = 0
            for k in range(1, N+1):
                sum += 0.5*delta*(self.calc_expected_tranche_principal(A,B,S,(k-1))-self.calc_expected_tranche_principal(A,B,S,k))*self.zcb[2*k-1]
            return sum
        
        return self.integrate_gauss_hermite(integrand)
    
    def calc_PV_expected_payoffs(self,A,B):  #C in Hull
        delta = self.delta # assume equidistant payments
        def integrand(S): #conditional PV_expected_payoffs * density
            N = self.N
            sum = 0
            for k in range(1, N+1):
                sum += (self.calc_expected_tranche_principal(A,B,S,k-1)-self.calc_expected_tranche_principal(A,B,S,k))*self.zcb[2*k-1]
            
            return sum
        
        return self.integrate_gauss_hermite(integrand)
    
    def calc_fair_spread_and_upfront(self,A,B):
        fair_spread = float((self.calc_PV_expected_payoffs(A,B))/(self.calc_PV_expected_regular_spread_payments(A,B)+self.calc_accrual_payment(A,B)))
        if fair_spread > 0.01:
            spread = 0.01
            upfront = float(self.calc_PV_expected_payoffs(A,B)-spread*(self.calc_PV_expected_regular_spread_payments(A,B) + self.calc_accrual_payment(A,B)))
            return spread,upfront
        else:
            return fair_spread,0

    def tranche_spread_for_rho(self, A, B, rho):

        old_rho = self.rho
        self.rho = rho

        C = self.calc_PV_expected_payoffs(A, B)
        P = (
            self.calc_PV_expected_regular_spread_payments(A, B)
            + self.calc_accrual_payment(A, B)
        )

        self.rho = old_rho

        return C / P
    
    def calc_compound_correlation(self, A, B, s, U):
        def objective(rho):
            def f(rho):
                old_rho = self.rho
                self.rho = rho

                C = self.calc_PV_expected_payoffs(A, B)
                P = (
                    self.calc_PV_expected_regular_spread_payments(A, B)
                    + self.calc_accrual_payment(A, B)
                )

                self.rho = old_rho

                return U + s * P - C
            return (f(rho))**2

        res = minimize_scalar(objective, bounds=(0.0001,0.9999), method='bounded')
        return res.x
    
    def calc_base_correlation(self, target_aq, market_quotes):
        total_loss_0_aq = 0.0
        
        for (A, B, s, U) in market_quotes:
            if B > target_aq:
                break
                
            comp_rho = self.calc_compound_correlation(A, B, s, U)
            
            old_rho = self.rho
            self.rho = comp_rho
            
            C_q_absolute = self.calc_PV_expected_payoffs(A, B) * (B - A)
            total_loss_0_aq += C_q_absolute
            
            self.rho = old_rho
            
        def objective(rho):
            old_rho = self.rho
            self.rho = rho
            model_loss = self.calc_PV_expected_payoffs(0, target_aq) * target_aq
            self.rho = old_rho
            return model_loss - total_loss_0_aq

        return brentq(objective, 0.0001, 0.9999)
    

sim = GaussianCopula(
    M = 125,
    R = 0.4,
    rho = 0.5,
    T = 5,
    delta = 1/4,
    n_herm=200)

market_info = [
    (0.00, 0.03, 0.01, 0.26346),
    (0.03, 0.06, 0.01, 0.03927),
    (0.06, 0.12, 0.009806, 0.0),
    (0.12, 1.00, 0.002678, 0.0)
]
sim.calc_base_correlation(0.3,market_info)

[0.         0.00021485 0.00042966 0.00064442 0.00085913 0.00182011
 0.00278017 0.00428321 0.00578397 0.00791029 0.01003205 0.0128937
 0.01574708 0.01923151 0.0227036  0.02661352 0.03050779 0.03352122
 0.03652528 0.04158012 0.04660844]


0.9990588948615075